## Data Preparation
-----

This notebook focuses on generating **nested training datasets** based on total audio duration.

We create progressively larger subsets (e.g., 1h → 2h → 5h → 10h), where each dataset includes all samples from the previous one. This supports consistent and reproducible training experiments.

- Uses a single shuffled manifest  
- Builds subsets using cumulative audio duration  
- Outputs separate CSV manifests for each duration  

> **Note:** Audio files are **not copied**. All subsets reference the same underlying audio paths, making the process fast and storage-efficient.


## 1. Python Setup

In [21]:
from pathlib import Path
from collections import namedtuple
import pandas as pd
import numpy as np
from numpy import random
import shutil
import os
import sys
import traceback
import re
from pydub import AudioSegment
from pydub.utils import make_chunks

from datetime import datetime
import librosa
import soundfile as sf

# Add the parent directory to the system path to allow imports from src
DIR_PARENT = Path.cwd().parent
sys.path.append(str(DIR_PARENT))
from src.data_utils.data_utils import sample_by_duration

/Users/dmatekenya/git-repos/chichewa-asr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Input Folder Configuration

In [2]:
# Base data directory
DIR_DATA = Path.cwd().parents[0].joinpath('data')

# Direcotory for test data and manifest file
DIR_TEST = DIR_DATA / "test"
FILE_MANIFEST_TEST = DIR_TEST / "metadata.csv"

# Directory for dev data and manifest file
DIR_DEV = DIR_DATA / "dev"
FILE_MANIFEST_DEV = DIR_DEV / "metadata.csv"

# Directory for nested duration based data
DIR_DEV_NESTED_DURATION = DIR_DATA / "dev_nested_duration"


## 3. Define Utility Functions 

In [12]:
def create_nested_duration_datasets(
    input_folder,
    output_folder,
    manifest_name="manifest.csv",
    audio_col="audio_fname",
    duration_col="duration",
    start_hours=1,
    max_hours=10,
    step_hours=1,
    seed=42,
    shuffle=True,
):
    """
    Create nested duration-based ASR datasets.

    For example, if start_hours=1 and max_hours=10, this creates:
    train_1h.csv, train_2h.csv, ..., train_10h.csv

    Each larger dataset contains all rows from the previous smaller dataset.

    Parameters
    ----------
    input_folder : str or Path
        Folder containing the input manifest file.

    output_folder : str or Path
        Folder where nested manifests will be written.

    manifest_name : str
        Name of the input manifest file.

    audio_col : str
        Column containing audio file paths.

    duration_col : str
        Column containing audio duration in seconds.

    start_hours : int or float
        First nested dataset duration in hours.

    max_hours : int or float
        Maximum nested dataset duration in hours.

    step_hours : int or float
        Interval between nested datasets in hours.

    seed : int
        Random seed for reproducible shuffling.

    shuffle : bool
        Whether to shuffle before creating nested subsets.

    Returns
    -------
    dict
        Summary of generated datasets.
    """

    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    manifest_path = input_folder / manifest_name
    df = pd.read_csv(manifest_path)

    if duration_col not in df.columns:
        raise ValueError(f"Missing duration column: {duration_col}")

    if audio_col not in df.columns:
        raise ValueError(f"Missing audio column: {audio_col}")

    df = df.dropna(subset=[audio_col, duration_col]).copy()
    df[duration_col] = pd.to_numeric(df[duration_col], errors="coerce")
    df = df.dropna(subset=[duration_col])
    df = df[df[duration_col] > 0].copy()

    if shuffle:
        df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    df["cumulative_duration_seconds"] = df[duration_col].cumsum()
    df["cumulative_duration_hours"] = df["cumulative_duration_seconds"] / 3600

    summaries = {}

    duration_targets = np.arange(start_hours, max_hours + step_hours, step_hours)

    for target_hours in duration_targets:
        subset = df[df["cumulative_duration_hours"] <= target_hours].copy()

        if subset.empty:
            continue

        output_name = f"train_{target_hours:g}h.csv"
        output_path = output_folder / output_name

        subset = subset.drop(
            columns=["cumulative_duration_seconds", "cumulative_duration_hours"]
        )

        subset.to_csv(output_path, index=False)

        summaries[output_name] = {
            "num_files": len(subset),
            "total_hours": round(subset[duration_col].sum() / 3600, 3),
            "output_path": str(output_path),
        }

    return summaries

## 4. Run the Data Generation

In [ ]:
# ============================================
# CREATE NESTED DURATION DATASETS FOR DEV SET
# =============================================
summaries = create_nested_duration_datasets(
    input_folder=DIR_DEV,
    output_folder=DIR_DEV_NESTED_DURATION,
    manifest_name="metadata.csv",
    audio_col="audio_fname",
    duration_col="duration",
    start_hours=1,
    max_hours=10,
    step_hours=1,
    seed=42,
)


## 5. Sampling Data 


In [24]:
df_all = pd.read_csv(FILE_MANIFEST_DEV)

In [25]:
df_10h = sample_by_duration(
    df=df_all,
    target_hours=10,
    random_state=42
)